# 1. A look ahead 
**Programming (5AF2029)** · Week 1 · Université de Neuchâtel · Dr. Selena Baset

[Course home](index.ipynb)  ·  [Next](02_setup.ipynb)

---

This first notebook is a demonstration, not a lesson. Do not try to understand the code today. Run each cell with **Shift + Enter** and answer the questions as they come.

## First demo: Would you play?

I offer you a game.

You start with **100 CHF**. I flip a fair coin.

- **Heads**: your entire wealth increases by 50 percent
- **Tails**: your entire wealth decreases by 40 percent

We play **100 rounds**. Each round applies to whatever you hold at that moment.

Before anything else: would you play?

### The textbook answer

One round multiplies your wealth by 1.5 or by 0.6, with equal probability. The expected multiplier is therefore

$$0.5 \times 1.5 + 0.5 \times 0.6 = 1.05$$

A gain of 5 percent per round, in expectation. Over 100 rounds that compounds.

**Question 1.** Guess what 100 CHF becomes after 100 rounds.

In [ ]:
guess = 0        # <- your guess, in CHF

In [ ]:
expected = 100 * 1.05 ** 100

print(f"You guessed  {guess:>12,} CHF")
print(f"Expectation  {expected:>12,.0f} CHF")

Thirteen thousand francs from a hundred. The bet looks not merely good but extraordinary.

## So let us play it

One player, one hundred rounds.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)

wealth = 100.0
for round_number in range(100):
    if rng.random() < 0.5:
        wealth = wealth * 1.5      # heads
    else:
        wealth = wealth * 0.6      # tails

print(f"Final wealth: {wealth:,.2f} CHF")

Fifty-two centimes.

That player was not even unlucky: they got 50 heads out of 100, exactly the fair split.

Run the cell below several times. It plays the same game with a fresh coin each time, and also reports how many heads came up.

In [ ]:
wealth = 100.0
heads_count = 0

for round_number in range(100):
    if np.random.random() < 0.5:
        wealth = wealth * 1.5
        heads_count = heads_count + 1
    else:
        wealth = wealth * 0.6

print(f"Heads: {heads_count} out of 100")
print(f"Final wealth: {wealth:,.2f} CHF")

You need roughly **56 heads out of 100** just to break even. A fair coin gives you 50.

That is the whole trick, and it is not hidden anywhere: a 50 percent gain and a 40 percent loss do not cancel. One of each, in either order, leaves you with 90 percent of what you had.

## Ten thousand players

One player proves nothing. Let us run ten thousand of them, each playing their own hundred rounds.

**Question 2.** Out of 10,000 players, what share do you think end with **less than the 100 CHF they started with**?

In [ ]:
guess = 0        # <- your guess, as a percentage

In [ ]:
players, rounds = 10_000, 100
rng = np.random.default_rng(2026)

heads = rng.random((players, rounds)) < 0.5
paths = 100 * np.cumprod(np.where(heads, 1.5, 0.6), axis=1)
final = paths[:, -1]

print(f"You guessed {guess}%")
print(f"Ended with less than they started: {100 * (final < 100).mean():.1f}%")
print(f"Ended with less than 1 CHF:        {100 * (final < 1).mean():.1f}%")

Here is what those ten thousand lives look like. The scale is logarithmic, so each step up the axis is ten times richer.

In [ ]:
plt.figure(figsize=(10, 5))

for i in range(20):
    plt.plot(paths[i], color="steelblue", alpha=0.5, linewidth=1)

plt.plot(paths.mean(axis=0), color="crimson", linewidth=2.5, label="average of all players")
plt.plot(np.median(paths, axis=0), color="darkorange", linewidth=2.5, label="the middle player")
plt.axhline(100, color="black", linestyle="--", linewidth=1, label="starting wealth")

plt.yscale("log")
plt.xlabel("round")
plt.ylabel("wealth, CHF")
plt.title("Twenty players, and what the whole population did")
plt.legend()
plt.show()


**Question 3.** The red line goes up and the orange line goes down, and both describe the same ten thousand players. What is each one telling you?

Both are right. And you saw it in a couple of seconds. Python is a very useful tool for this: a few lines of code turn data into a picture, and you can see the "big picture" straight away.

# Second demo: A line that fills a square

The first example put a number on an argument. This one is here because it is beautiful, and because it is the shortest demonstration I know that a very small rule can have very large consequences.

In 1891 David Hilbert asked a question that sounds like nonsense. A line has one dimension. A square has two. Can a single line, drawn without lifting the pen and without crossing itself, pass through **every point** of a square?

He answered it by writing down a rule. The rule is:

> Take a square and cut it into four quadrants. Draw a smaller version of the same line in each quadrant, turn two of them so the ends meet, and join them up. Each of those smaller versions is built by doing the same thing again.

NOW, Let's try to program that rule!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def hilbert(x, y, xi, xj, yi, yj, depth, points):
    if depth == 0:
        points.append((x + (xi + yi) / 2, y + (xj + yj) / 2))
        return

    hilbert(x, y,
            yi / 2, yj / 2, xi / 2, xj / 2, depth - 1, points)
    hilbert(x + xi / 2, y + xj / 2,
            xi / 2, xj / 2, yi / 2, yj / 2, depth - 1, points)
    hilbert(x + xi / 2 + yi / 2, y + xj / 2 + yj / 2,
            xi / 2, xj / 2, yi / 2, yj / 2, depth - 1, points)
    hilbert(x + xi / 2 + yi, y + xj / 2 + yj,
            -yi / 2, -yj / 2, -xi / 2, -xj / 2, depth - 1, points)

This is what we cann a reccursive function. The cell above defines the hilbert() functiona **and** makes four calls to the function being defined. 


Let's see how this works: 


At depth 1 the curve is about as unimpressive as a curve can be.

In [ ]:
points = []
hilbert(0, 0, 1, 0, 0, 1, 1, points)
points = np.array(points)

plt.figure(figsize=(4, 4))
plt.plot(points[:, 0], points[:, 1], linewidth=2)
plt.axis("equal"); plt.axis("off")
plt.show()

Three strokes. Now watch the same rule at depths 1 to 6. Nothing changes except one number.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 7))

for ax, depth in zip(axes.ravel(), range(1, 7)):
    points = []
    hilbert(0, 0, 1, 0, 0, 1, depth, points)
    points = np.array(points)

    ax.plot(points[:, 0], points[:, 1], linewidth=2.4 - 0.32 * depth)
    ax.set_title(f"depth {depth}   ({len(points):,} points)", fontsize=10)
    ax.set_aspect("equal"); ax.axis("off")

plt.tight_layout()
plt.show()


**Question 4.** The square is 1 unit wide. At depth 7, how long is the line inside it?

In [ ]:
guess = 0        # <- your guess, in units

In [ ]:
points = []
hilbert(0, 0, 1, 0, 0, 1, 9, points)
points = np.array(points)

length = np.abs(np.diff(points, axis=0)).sum()

print(f"You guessed  {guess}")
print(f"Points       {len(points):,}")
print(f"Line length  {length:.0f} units, inside a square 1 unit wide")

It doubles with every depth. 8, 16, 32, 64, 128, and on. Take the depth to infinity, which is what Hilbert actually meant, and you have a line of **infinite length** that fits inside a square of area 1, never crosses itself, and passes through every point in it.

Try changing the 7 to 8 and run it again. Then 9, if you are patient :)

### It is not only pretty

Colour the curve by the order in which it visits each point, early in blue, late in red.

In [ ]:
points = []
hilbert(0, 0, 1, 0, 0, 1, 6, points)
points = np.array(points)

plt.figure(figsize=(7, 7))
plt.scatter(points[:, 0], points[:, 1],
            c=range(len(points)), cmap="turbo", s=2)
plt.axis("equal"); plt.axis("off")
plt.show()

The colours come in solid blocks. That is the useful property: points that are close together in the square are visited at close together times, and points visited close together in time are close together in the square.

Now take a step back and observe: four lines of instruction in the function definition, and nobody ever said what the shape should be. You control the logic, the program does the rest. That's really all programming is.

---

# Third demo: Knowing when to give up

Part two used recursion to draw. This one uses it to **solve**.

Here is a Sudoku. The rules you know: every row, every column and every three by three box must contain each digit from 1 to 9 exactly once. Fifty-two of the eighty-one squares are empty.

In [ ]:
puzzle = [
    [0, 0, 4, 6, 0, 7, 8, 9, 0],
    [9, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 4, 0, 0, 3],
    [4, 0, 0, 0, 0, 0, 2, 0, 8],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 2, 7, 9, 0, 0, 5],
    [0, 0, 0, 0, 0, 2, 0, 0, 1],
    [0, 0, 1, 9, 0, 0, 5, 8, 7],
    [8, 7, 0, 3, 0, 0, 0, 2, 4],
]


def show(grid):
    for r in range(9):
        if r % 3 == 0 and r > 0:
            print("------+-------+------")
        line = ""
        for c in range(9):
            if c % 3 == 0 and c > 0:
                line = line + "| "
            line = line + (str(grid[r][c]) if grid[r][c] else ".") + " "
        print(line)


show(puzzle)

**Question 5.** Fifty-two blanks, nine possible digits in each. How many ways are there to fill them in? Guess the order of magnitude: is it thousands, millions, more?

In [ ]:
possibilities = 9 ** 52

print(f"Ways to fill the blanks: {possibilities:.2e}")

seconds_since_big_bang = 13.8e9 * 365.25 * 24 * 3600
checked = seconds_since_big_bang * 1e9      # a billion per second, since the beginning

print(f"A machine checking a billion per second since the Big Bang: {checked:.2e}")
print(f"It would still be short by a factor of {possibilities / checked:.0e}")

So trying every combination isn't slow, it's impossible, and no faster computer will ever fix that. We haven't even attempted the puzzle yet. We've only counted the possibilities, and that's already enough to rule out brute force.

### The idea

A human solving a Sudoku does not try combinations. They put a digit in a square, and if it leads to a contradiction they **rub it out and try the next one**.

Written as a procedure:

1. Find the first empty square.
2. Try 1. If it does not clash with the row, column or box, write it in and solve the rest of the grid the same way.
3. If the rest of the grid turns out to be unsolvable, rub out the digit and try 2, then 3, and so on.
4. If none of the nine work, this square cannot be filled: give up and report failure to whoever asked.

Step 2 solves a smaller version of the same problem, so this is recursion again. Step 3 is the part that makes it work: **the program undoes its own decisions.** That is called **backtracking**.

In [ ]:
def is_allowed(grid, row, col, digit):
    if digit in grid[row]:
        return False
    if digit in [grid[r][col] for r in range(9)]:
        return False

    box_row, box_col = 3 * (row // 3), 3 * (col // 3)
    for r in range(box_row, box_row + 3):
        for c in range(box_col, box_col + 3):
            if grid[r][c] == digit:
                return False
    return True

In [ ]:
attempts = 0


def solve(grid):
    global attempts

    for row in range(9):
        for col in range(9):
            if grid[row][col] == 0:

                for digit in range(1, 10):
                    if is_allowed(grid, row, col, digit):
                        grid[row][col] = digit      # write it in
                        attempts = attempts + 1

                        if solve(grid):             # solve the rest
                            return True

                        grid[row][col] = 0          # rub it out again

                return False        # no digit fits here: this path is dead

    return True                     # no empty squares left: solved

Twenty lines, and not one of them contains a Sudoku technique. No naked pairs, no hidden singles, nothing a puzzle book would teach you. Run it.

In [ ]:
import copy, time

grid = copy.deepcopy(puzzle)
attempts = 0

start = time.time()
solve(grid)
elapsed = time.time() - start

show(grid)
print()
print(f"Solved in {elapsed:.3f} seconds")
print(f"Digits written in and rubbed out again: {attempts:,}")

A few thousand attempts, against the 4 × 10⁴⁹ combinations that exist.

Nothing was skipped by luck. The program never examined the overwhelming majority of those combinations because it noticed, early, that they all began with a contradiction. A single wrong digit in the top row invalidates an unimaginable number of grids at once, and the program throws all of them away the moment it sees the clash.

**The intelligence is not in knowing the answer. It is in recognising a dead end and going back.** That single idea, try something, check, undo if it fails, is enough to solve timetabling, routing, portfolio construction under constraints, and a large share of the problems that look hopeless when you first write them down.

---

# Fourth demo: What the numbers do not say

The three examples so far computed something. This one does the opposite: it is about what happens when you trust a computation and never look at the data.

A colleague sends you four datasets and asks whether they are alike. Each is eleven pairs of measurements.

In [ ]:
import pandas as pd

x_common = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]

data = pd.DataFrame({
    "dataset": ["I"] * 11 + ["II"] * 11 + ["III"] * 11 + ["IV"] * 11,
    "x": x_common + x_common + x_common
         + [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
    "y": [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]
         + [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]
         + [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]
         + [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
})

data.head()

You would do what anyone does: summarise them. Average, spread, and how strongly `x` and `y` move together.

In [ ]:
summary = data.groupby("dataset").agg(
    mean_x=("x", "mean"),
    mean_y=("y", "mean"),
    sd_x=("x", "std"),
    sd_y=("y", "std"),
)

summary["correlation"] = data.groupby("dataset")[["x", "y"]].corr().unstack()[("y", "x")]

summary.round(2)

Every column is the same, to two decimal places, in all four datasets. Same centre, same spread, same correlation. Fit a straight line through each and you get the same line as well: `y = 3 + 0.5x`.

On that evidence you would report that the four datasets are interchangeable.

**Question 6.** Are they?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True)

for ax, name in zip(axes.ravel(), ["I", "II", "III", "IV"]):
    subset = data[data["dataset"] == name]

    ax.scatter(subset["x"], subset["y"], s=45)
    ax.plot([3, 20], [3 + 0.5 * 3, 3 + 0.5 * 20], color="crimson", linewidth=1)
    ax.set_title(f"dataset {name}")

plt.tight_layout()
plt.show()

One is an ordinary cloud. One is a perfect curve, and a straight line is simply the wrong model for it. One is a tight straight relationship ruined by a single outlier. One has no relationship at all: ten identical readings and one point far away, which alone produces the entire correlation.

Four completely different situations, calling for four completely different responses, and every summary statistic agreed they were the same.

### Why this matters more than the other three examples

These are Anscombe's quartet, constructed in 1973 to make exactly this point. They are artificial. The phenomenon is not.

A summary statistic is a compression, and compression discards. An average return hides whether the gains came steadily or in one month. A correlation hides whether it rests on the whole sample or on a single crisis. A model fit hides whether the model was the right shape in the first place.

The chart cost four lines. It is not decoration and it is not the last step of the analysis. It is how you find out whether the analysis was worth doing.

---

## What you've seen are four modes of teaching the machine to solve a problem, aka to program!

| | The move |
|---|---|
| **Simulation** | Do it ten thousand times and see what actually happens |
| **Recursion** | Define a thing in terms of a smaller version of itself |
| **Backtracking** | Try, check, undo, try the next |
| **Visualising** | Draw it, because the numbers agreed and the pictures did not |

Don't worry much about the specific algorithms used in these examples. They are not the focus of the course.

The point is that programming is less about the syntax and more about how you abstract a problem and dictate to the machine the logic to solve it.

Everything you ran today is built from variables, conditions, loops and functions. That is what we cover, and you will have all four by week 9.

If any of the algorithms interests you, feel free to spend more time on the example and come and ask if you have questions.

---

**Next:** [Setting up](02_setup.ipynb)